In [4]:
import os
import sys
from pathlib import Path

# Configurar AWS profile com credenciais (evita SSO + SSL issues)
os.environ['AWS_PROFILE'] = '759242759842_CONSUMER'

# CA bundle corporativo (necessário para o proxy Itaú não causar CERTIFICATE_VERIFY_FAILED)
home = Path.home()
ca_paths = [
    home / '.aws' / 'cacert.pem',  # CA Itaú
    home / '.aws' / 'cacert-987979f15e8bd2c573161b23c2885fda.crt',  # CA alternativo
]

ca_bundle = None
for ca_path in ca_paths:
    if ca_path.exists():
        ca_bundle = str(ca_path)
        os.environ['AWS_CA_BUNDLE']      = ca_bundle
        os.environ['REQUESTS_CA_BUNDLE'] = ca_bundle
        os.environ['CURL_CA_BUNDLE']     = ca_bundle
        print(f'✅ CA bundle configurado: {ca_bundle}')
        break

if not ca_bundle:
    print('⚠️  AVISO: nenhum CA bundle encontrado em ~/.aws/')
    print('    A conexão pode falhar com erro SSL')

# Proxy corporativo (se necessário)
os.environ['HTTP_PROXY']  = 'http://proxynew.itau:8080'
os.environ['HTTPS_PROXY'] = 'http://proxynew.itau:8443'

print(f'AWS Profile: {os.environ.get("AWS_PROFILE")}')
print(f'CA Bundle: {ca_bundle or "não definido"}')
print(f'HTTPS Proxy: {os.environ.get("HTTPS_PROXY")}')

✅ CA bundle configurado: C:\Users\ulgczaq\.aws\cacert.pem
AWS Profile: 759242759842_CONSUMER
CA Bundle: C:\Users\ulgczaq\.aws\cacert.pem
HTTPS Proxy: http://proxynew.itau:8443


In [5]:
import awswrangler as wr
import boto3

# Testar conexão com Athena
print('Testando conexão com Athena...')
try:
    client = boto3.client('athena', region_name='sa-east-1')
    response = client.list_work_groups()
    print(f'✅ Conexão OK — {len(response["WorkGroups"])} workgroups encontrados')
except Exception as e:
    print(f'❌ Erro na conexão: {e}')
    sys.exit(1)

# Configuração da query
QUERY    = 'SELECT * FROM database_rt2.RT2_AI6_OCORRENCIA_FQ_001 LIMIT 10'
DATABASE = 'database_rt2'
WORKGROUP = 'analytics-workgroup-v3'

print(f'\nExecutando query...\n{QUERY}\n')

try:
    df = wr.athena.read_sql_query(
        sql=QUERY,
        database=DATABASE,
        ctas_approach=False,
        workgroup=WORKGROUP,
    )
    print(f'✅ OK — {len(df)} linhas, {len(df.columns)} colunas')
    df.head()
except Exception as e:
    print(f'❌ Erro na query: {e}')
    import traceback
    traceback.print_exc()

Testando conexão com Athena...
✅ Conexão OK — 3 workgroups encontrados

Executando query...
SELECT * FROM database_rt2.RT2_AI6_OCORRENCIA_FQ_001 LIMIT 10

✅ OK — 10 linhas, 104 colunas
